# 🛒 NexoMarket Decision Room — Semana 8
## Web → Data Factory → SQL Server → SELECT avanzado → Decisión → Power BI

**Rol:** Data Analyst Jr. + SQL Developer Jr.  
**Duración total del laboratorio:** 2 h 30 min.

Hoy no construiremos una web. Obtendremos **productos de una API pública de prueba**, los convertiremos en un dataset operacional de NexoMarket y llevaremos los datos a SQL Server.

> ⚠️ Los productos base provienen de DummyJSON, una API de prueba para prototipos. Los campos operacionales (tienda, costo, ventas, stock, proveedor, etc.) serán **simulados con fines académicos**; no representan ventas reales.


## Checkpoint 0 — Traer datos desde la web
La API de productos incluye campos como título, categoría, precio, rating, stock, marca y SKU.  
Ejecuta la siguiente celda. Si el acceso web del laboratorio falla, el notebook creará automáticamente una base de respaldo para que la práctica continúe.


In [ ]:
import pandas as pd
import numpy as np
import requests, os, json
np.random.seed(42)

URL = "https://dummyjson.com/products?limit=0"
try:
    r = requests.get(URL, timeout=20)
    r.raise_for_status()
    productos_web = r.json()["products"]
    fuente = "DummyJSON API"
    print("✅ Datos obtenidos desde la web:", len(productos_web), "productos base")
except Exception as e:
    print("⚠️ No hubo acceso a la API. Se usará respaldo académico.")
    categorias=["laptops","tablets","smartphones","mobile-accessories","sports-accessories"]
    marcas=["Lenovo","HP","Dell","ASUS","Acer","Samsung","Apple","Xiaomi"]
    productos_web=[]
    for i in range(1,201):
        c=np.random.choice(categorias); m=np.random.choice(marcas)
        productos_web.append({"id":i,"title":f"{m} {c} Pro {i}",
                              "category":c,"price":float(np.random.randint(80,1200)),
                              "rating":round(float(np.random.uniform(2.5,5)),1),
                              "stock":int(np.random.randint(0,100)),"brand":m,"sku":f"SKU{i:04d}"})
    fuente="Respaldo simulado"
print("Fuente:", fuente)
pd.DataFrame(productos_web).head()


## Checkpoint 1 — Data Factory: convertir productos base en 50 000 ofertas
Cada producto base puede aparecer como una **oferta comercial** en distintas sedes/proveedores. Esto permite trabajar con un volumen suficiente sin afirmar que existen 50 000 productos reales en la web.

Se crean campos útiles para decisiones: `precio`, `costo`, `stock`, `ventas30dias`, `proveedor`, `ciudad`, `estado`, `rating`, `segmento`.


In [ ]:
base_df = pd.DataFrame(productos_web)
N=50000
idx=np.random.choice(base_df.index, N, replace=True)
src=base_df.loc[idx].reset_index(drop=True)

ciudades=["Lima","Arequipa","Trujillo","Chiclayo","Piura","Cusco"]
proveedores=["TechImport SAC","Distribuciones Andinas","Global Devices","Pacific Tech","CompuMayorista",None]

precio_usd=pd.to_numeric(src.get("price",100), errors="coerce").fillna(100)
# Tipo de cambio SIMULADO para el ejercicio, no es una cotización financiera.
precio=(precio_usd*3.8*np.random.uniform(.82,1.18,N)).round(2)
costo=(precio*np.random.uniform(.58,.82,N)).round(2)
stock=np.random.poisson(24,N)
stock[np.random.choice(N,3200,replace=False)]=0
ventas=np.random.negative_binomial(5,.20,N)
rating=pd.to_numeric(src.get("rating",4),errors="coerce").fillna(4).round(1)
rating.iloc[np.random.choice(N,1800,replace=False)] = np.nan

df=pd.DataFrame({
 "idOferta":range(1,N+1),
 "idProductoWeb":src["id"].values,
 "nombre":src["title"].astype(str).values,
 "categoria":src["category"].astype(str).str.replace("-"," ",regex=False).str.title().values,
 "marca":src.get("brand",pd.Series(["Sin marca"]*N)).fillna("Sin marca").astype(str).values,
 "sku":src.get("sku",pd.Series(["N/A"]*N)).fillna("N/A").astype(str).values,
 "precio":precio,
 "costo":costo,
 "stock":stock,
 "ventas30dias":ventas,
 "rating":rating.values,
 "proveedor":np.random.choice(proveedores,N,p=[.20,.18,.18,.17,.20,.07]),
 "ciudad":np.random.choice(ciudades,N,p=[.55,.10,.10,.08,.09,.08]),
 "estado":np.random.choice(["ACTIVO","ACTIVO","ACTIVO","ACTIVO","DESCONTINUADO"],N)
})
df["valorInventario"]=(df.precio*df.stock).round(2)
print(df.shape)
display(df.sample(8, random_state=8))


In [ ]:
# Calidad y perfil rápido
print("Registros:",len(df))
print("Categorías:",df.categoria.nunique())
print("Marcas:",df.marca.nunique())
print("Agotados:",(df.stock==0).sum())
print("Sin proveedor:",df.proveedor.isna().sum())
print("Sin rating:",df.rating.isna().sum())
display(df[["categoria","marca","precio","stock","ventas30dias","rating"]].describe(include="all"))


In [ ]:
# Descargar CSV para SQL Server
archivo="productos_nexomarket_50000.csv"
df.to_csv(archivo,index=False,encoding="utf-8-sig")
print("✅ Archivo creado:",archivo,round(os.path.getsize(archivo)/1024/1024,2),"MB")
try:
    from google.colab import files
    files.download(archivo)
except:
    pass


# 🗄️ Ahora pasamos a SQL Server
Importa el CSV como tabla `PRODUCTO`.

Valida:
```sql
SELECT COUNT(*) AS totalRegistros FROM PRODUCTO;
SELECT TOP 10 * FROM PRODUCTO;
```

## SQL Challenge — 8 retos
1. **AND / OR:** productos activos y con stock; luego compara con una condición OR.
2. **IN:** categorías Laptop, Tablet o Smartphone.
3. **BETWEEN:** precios entre S/1500 y S/3000.
4. **LIKE:** nombres que contengan `Pro`; prueba también inicio y final.
5. **NULL:** productos sin proveedor; explica por qué `= NULL` no es correcto.
6. **DISTINCT:** marcas existentes sin repetir.
7. **TOP + ORDER BY:** 10 productos más caros disponibles.
8. **CASE:** crea `segmentoPrecio` y `estadoStock`.

### Boss Level
Obtén **10 productos** Laptop o Tablet, S/1500–S/3000, disponibles, con proveedor, cuyo nombre contenga `Pro`, no descontinuados, ordenados de mayor a menor y clasificados por precio.


# 🚨 Giro sorpresa: “Ahora entra el gerente”
Crea una VIEW `vw_DecisionRoom` que deje lista información para decisiones.

Debe contener, como mínimo:
- categoría, marca, precio, costo, stock, ventas30dias, rating y ciudad;
- `segmentoPrecio` mediante CASE;
- `estadoStock` mediante CASE;
- `valorInventario = precio * stock`;
- solo registros no descontinuados.

Después valida:
```sql
SELECT TOP 20 *
FROM vw_DecisionRoom
ORDER BY ventas30dias DESC;
```


# 📊 BONUS MISSION — Power BI Express (solo si llegaron a tiempo)
Conecta Power BI Desktop a SQL Server y selecciona `vw_DecisionRoom`.

Construye únicamente:
- **KPI 1:** productos con stock crítico.
- **KPI 2:** valor de inventario.
- **Gráfico 1:** ventas por categoría.
- **Gráfico 2:** productos con alta venta y bajo stock.
- **Segmentador:** categoría o marca.

### Gerente por 60 segundos
Completa:
> “Nuestro dashboard muestra ____________. Por eso recomendamos ____________.”

**SQL encuentra la información. Power BI la comunica. El negocio toma la decisión.**
